In [120]:
import datasets
import torch
import torch.nn as nn
import torch.nn.functional as f
import numpy as np
from torch.utils.data import Dataset, DataLoader
import math


In [ ]:

# Load the full dataset (all splits)
ds = datasets.load_dataset('tiny_shakespeare', trust_remote_code=True)

# Access individual splits
train_data = ds['train']
val_data = ds['validation']
test_data = ds['test']

# Print a sample from the train split
print(train_data[0]['text'][:500])  # First 500 characters of the text

Generating train split:   0%|          | 0/1 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1 [00:00<?, ? examples/s]

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


# Tokenizer

In [99]:

class CharTokenizer:
    def __init__(self, corpus):
        self.tokens = self.tokenize(corpus)
        self.vocabulary = ['<UNK>', '<PAD>'] + sorted(list(set(self.tokens)))
        self.vocab_size = len(self.vocabulary)
        self.stoi = {ch:i for i, ch in enumerate(self.vocabulary)}
        self.itos = {i:ch for i, ch in enumerate(self.vocabulary)}

    def str_2_int(self, text):
        return torch.tensor([self.stoi.get(ch, self.stoi['<UNK>']) for ch in text], dtype=torch.long)

    def int_2_str(self, encoded):
        encoded_list = encoded.numpy()  # Fast conversion
        return [self.itos.get(i, self.stoi['<UNK>']) for i in encoded_list]
    
    def get_data(self):
        return self.vocabulary, self.vocab_size
    
    def tokenize(self, corpus):
        return list(str(corpus['text'][0]).lower())


tokenizer = CharTokenizer(train_data)
vocabulary, vocab_size = tokenizer.get_data()

train_tokens = tokenizer.tokenize(train_data)
data_train = tokenizer.str_2_int(train_tokens)

val_tokens = tokenizer.tokenize(val_data)
data_val = tokenizer.str_2_int(val_tokens)

test_tokens = tokenizer.tokenize(test_data)
data_test = tokenizer.str_2_int(test_tokens)



# tokenizer.int_2_str(data_train)
vocab_size

41

In [98]:
x = [1, 2, 3, 4, 5, 6, 7, 8 , 9]

len(x)-4

5

# Data Pipeline

In [106]:

class GPT_DataSet(Dataset):
    def __init__(self, corpus, block_size):
        self.corpus = corpus
        self.block_size = block_size
    
    def __len__(self):
        return len(self.corpus) - self.block_size

    def __getitem__(self, i):
        return self.corpus[i:i+self.block_size], self.corpus[i+1:i+1+self.block_size]


block_size = 4
batch_size = 32
train_dataset = GPT_DataSet(data_train, block_size)
val_dataset = GPT_DataSet(data_val, block_size)
test_dataset = GPT_DataSet(data_test, block_size)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# train_loader[0], val_loader[0], test_loader[0]

# GPT Implementation

In [ ]:

class Simple_GPT(nn.Module):
    def __init__(self, vocab_size, embed_dim, block_size):
        super().__init__()
        self.token_embeddings = nn.Embedding(vocab_size, embed_dim)
        self.pos_embeddings = self.pos_encoding(embed_dim, block_size)

    def pos_encoding(self, embed_dims, block_size, type='sinusoidal'):
        if type == 'sinusoidal':
            pe = torch.zeros(block_size, embed_dims)
            for pos in range(block_size):
                for i in range(0, embed_dims, 2):
                    angle = pos / (10000**(i/embed_dims))
                    pe[pos, i] = math.sin(angle)

                    if i+1 < embed_dims:
                        pe[pos, i+1] = math.cos(angle)

            return pe.unsqueeze(0)

        else:
            return nn.Embedding(block_size, embed_dims)
        

    def layer_norm(self):
        pass

    def single_head_attention(self):
        pass

    def multi_head_attention(self):
        pass

    def feed_forward(self):
        pass

    def forward(self, x):
        
        input_embeddings = self.token_embeddings(x) + self.pos_embeddings

        return input_embeddings


embed_dim=128
gpt = Simple_GPT(vocab_size, embed_dim, block_size)

In [138]:
# Sample input: A batch of 2 sequences, each of length block_size
sample_indices = torch.randint(0, len(data_train) - block_size, (2,))  # Random starting points
sample_x = torch.stack([data_train[i:i+block_size] for i in sample_indices])  # Shape: (2, block_size)

print("Sample input shape:", sample_x.shape)  # e.g., torch.Size([2, 4])
print("Sample input:\n", sample_x)

# Instantiate model (as in your code)
embed_dim = 128
gpt = Simple_GPT(vocab_size, embed_dim, block_size)

# Forward pass
with torch.no_grad():  # No gradients for testing
    output = gpt(sample_x)

print("\nOutput shape:", output.shape)  # Should be (batch_size, block_size, embed_dim) = (2, 4, 128)
print("Sample output (first sequence, first 5 dims):\n", output[0, :, :5])  # Peek at embeddings

Sample input shape: torch.Size([2, 4])
Sample input:
 tensor([[23, 29, 26, 23],
        [20, 26, 19, 18]])

 torch.Size([2, 4, 128])

Output shape: torch.Size([2, 4, 128])
Sample output (first sequence, first 5 dims):
 tensor([[ 0.3566,  2.9800, -1.1165, -0.0652,  1.6024],
        [ 2.3685, -0.4186,  0.7333,  0.4863, -0.3656],
        [ 1.2228, -1.4842, -0.3673, -1.2059,  1.4342],
        [ 0.4977,  0.9900, -0.5992, -1.9210,  2.3807]])


# Training Loop